## Notebook 2: Data Cleaning & Feature Engineering

### This Notebook
Building on the raw award data acquired in Notebook 1, this notebook prepares the two core data structures that power the matching pipeline.

The central insight driving the feature engineering here is that grant program profiles are always dense, org profiles are often sparse. 43% of organizations in the dataset have fewer than 200 characters of combined description text — not enough to embed meaningfully. Grant programs, by contrast, aggregate descriptions across hundreds of recipients and consistently produce rich, embeddable text.

This motivates the matching architecture: rather than embedding org profiles directly, I embed grant program profiles and match a new org's self-description against them at query time.

**This notebook produces two outputs:**
- `grant_profiles.csv` — one row per CFDA program, with cleaned embedding text ready for Notebook 3
- `org_profiles_clean.csv` — one row per recipient organization, with parsed fields and simplified type labels

**Inputs:** `grant_recipients_2023.csv`, `org_profiles.csv` (from Notebook 1)

In [1]:
import pandas as pd
import numpy as np
import ast
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/grant_finder'

# Load the outputs from Notebook 1
df_clean = pd.read_csv(f'{DRIVE_PATH}/grant_recipients_2023.csv')
org_profiles = pd.read_csv(f'{DRIVE_PATH}/org_profiles.csv')
df_clean = df_clean[df_clean['federal_action_obligation'] > 0].copy()

Mounted at /content/drive


In [2]:
df_clean.shape
org_profiles.shape
df_clean.columns.tolist()

['federal_action_obligation',
 'period_of_performance_start_date',
 'period_of_performance_current_end_date',
 'awarding_agency_name',
 'recipient_uei',
 'recipient_name',
 'recipient_city_name',
 'recipient_state_code',
 'recipient_zip_code',
 'primary_place_of_performance_state_name',
 'cfda_number',
 'cfda_title',
 'assistance_type_description',
 'transaction_description',
 'business_types_description',
 'desc_length']

In [3]:
grant_profiles = df_clean.groupby(['cfda_number', 'cfda_title']).agg(
    awarding_agency=('awarding_agency_name', 'first'),
    num_recipients=('recipient_uei', 'nunique'),
    total_awarded=('federal_action_obligation', 'sum'),
    avg_award=('federal_action_obligation', 'mean'),
    min_award=('federal_action_obligation', 'min'),
    max_award=('federal_action_obligation', 'max'),
    org_types=('business_types_description', lambda x: list(x.value_counts().head(3).index)),
    states_funded=('recipient_state_code', lambda x: list(x.unique())),
    all_descriptions=('transaction_description', lambda x: ' '.join(x.dropna().unique()))
).reset_index()

grant_profiles['desc_length'] = grant_profiles['all_descriptions'].str.len()
grant_profiles.shape

(1222, 12)

In [4]:
grant_profiles['desc_length'].describe()

,desc_length
count,1.222000e+03
mean,2.147738e+05
std,1.296915e+06
min,5.000000e+00
25%,7.102500e+02
50%,4.445500e+03
75%,2.443825e+04
max,1.657279e+07


In [5]:
pd.Series({
    'programs < 200 chars': (grant_profiles['desc_length'] < 200).sum(),
    'programs > 1000 chars': (grant_profiles['desc_length'] > 1000).sum()
})

,0
programs < 200 chars,180
programs > 1000 chars,870


# Checking Description Lengths


In [6]:
sparse = grant_profiles[grant_profiles['desc_length'] < 200].sort_values('desc_length')

sparse[['cfda_number', 'cfda_title', 'num_recipients', 'all_descriptions']].head(10)

,cfda_number,cfda_title,num_recipients,all_descriptions
432,15.978,UPPER MISSISSIPPI RIVER RESTORATION LONG TERM ...,1,LTRMP
105,10.587,NATIONAL FOOD SERVICE MANAGEMENT INSTITUTE ADM...,1,CN NFSMI
615,20.111,AIRCRAFT PILOTS WORKFORCE DEVELOPMENT GRANT PR...,10,AWD PILOTS
254,12.113,STATE MEMORANDUM OF AGREEMENT PROGRAM FOR THE ...,1,DSMOA PROGRAM
879,89.003,NATIONAL HISTORICAL PUBLICATIONS AND RECORDS G...,68,DISCRETIONARY
616,20.112,AVIATION MAINTENANCE TECHNICAL WORKFORCE GRANT...,10,AWD MAINTENANCE
98,10.533,SNAP-ED TOOLKIT,1,SNAP-ED TOOLKIT
1017,93.382,INDIAN HEALTH SERVICE COMMUNITY HEALTH AIDE PR...,1,SPTHB TPI GRANT
868,84.380,SPECIAL EDUCATION - SPECIAL OLYMPICS EDUCATION...,1,SPECIAL OLYMPICS
101,10.545,FARMERS’ MARKET SUPPLEMENTAL NUTRITION ASSISTA...,1,SNAP EBT PROJECTS


In [7]:
giant = grant_profiles[grant_profiles['desc_length'] > 1_000_000]
giant[['cfda_number', 'cfda_title', 'num_recipients', 'desc_length']]

,cfda_number,cfda_title,num_recipients,desc_length
63,10.310,AGRICULTURE AND FOOD RESEARCH INITIATIVE (AFRI),181,1838226
682,47.041,ENGINEERING,305,8043842
683,47.049,MATHEMATICAL AND PHYSICAL SCIENCES,441,11661889
684,47.050,GEOSCIENCES,323,5090477
685,47.070,COMPUTER AND INFORMATION SCIENCE AND ENGINEERING,355,8119434
686,47.074,BIOLOGICAL SCIENCES,412,4878673
687,47.075,"SOCIAL, BEHAVIORAL, AND ECONOMIC SCIENCES",262,2833070
689,47.076,STEM EDUCATION (FORMERLY EDUCATION AND HUMAN R...,661,5360228
692,47.083,INTEGRATIVE ACTIVITIES,264,2235919
693,47.084,"NSF TECHNOLOGY, INNOVATION, AND PARTNERSHIPS",882,3421017


# Setting Char limit for descriptions

In [8]:
def build_embedding_text(row, max_chars=50000):
    """
    Build a clean text representation of a grant program for embedding.
    Falls back to title for sparse programs, truncates giant ones.
    """
    title = row['cfda_title']
    agency = row['awarding_agency']
    desc = row['all_descriptions']

    if len(desc) < 200:
        return f"{title}. Awarded by {agency}."

    if len(desc) > max_chars:
        desc = desc[:max_chars]

    return f"{title}. Awarded by {agency}. {desc}"

grant_profiles['embedding_text'] = grant_profiles.apply(build_embedding_text, axis=1)
grant_profiles['embedding_text_length'] = grant_profiles['embedding_text'].str.len()

grant_profiles['embedding_text_length'].describe()

,embedding_text_length
count,1222.000000
mean,14675.247136
std,18445.899676
min,47.000000
25%,801.500000
50%,4532.000000
75%,24559.250000
max,50175.000000


In [9]:
pd.Series({
    'using title fallback': (grant_profiles['desc_length'] < 200).sum(),
    'truncated': (grant_profiles['desc_length'] > 50000).sum()
})

,0
using title fallback,180
truncated,192


In [10]:
# Sparse sample
grant_profiles[grant_profiles['desc_length'] < 200]['embedding_text'].iloc[0]

'MARKET NEWS. Awarded by Department of Agriculture.'

In [11]:
# Normal sample
grant_profiles[(grant_profiles['desc_length'] >= 200) &
               (grant_profiles['desc_length'] < 50000)]['embedding_text'].iloc[0][:300]

'AGRICULTURAL RESEARCH BASIC AND APPLIED RESEARCH. Awarded by Department of Agriculture. ARS INTERACTION/SUPPORT OF HACU PROGRAMS CLIMATE CHANGE MITIGATION OUTREACH AND EDUCATION IN THE SOUTHERN PLAINS SUPPLEMENTAL CA 59-3000-1-001 TO UMBRELLA AGREEMENT 59-0101-0-158 EVALUATION OF NOVEL COLD HARDY GR'

# Profile Cleaning

In [12]:
import re
import numpy as np

def parse_agencies(val):
    if isinstance(val, list):
        return val
    if not isinstance(val, str):
        return []
    return re.findall(r"'([^']*)'", val)

def parse_cfda_numbers(val):
    if isinstance(val, list):
        return list(dict.fromkeys(val))
    if not isinstance(val, str):
        return []
    nums = re.findall(r'np\.float64\(([\d.]+)\)', val)
    if nums:
        return list(dict.fromkeys([float(n) for n in nums]))
    nums = re.findall(r'\b(\d+\.\d+)\b', val)
    return list(dict.fromkeys([float(n) for n in nums])) if nums else []

def parse_cfda_titles(val):
    if isinstance(val, list):
        return val
    if not isinstance(val, str):
        return []
    return re.findall(r"'([^']*)'", val)

org_profiles['agencies'] = org_profiles['agencies'].apply(parse_agencies)
org_profiles['cfda_programs'] = org_profiles['cfda_programs'].apply(parse_cfda_numbers)
org_profiles['cfda_titles'] = org_profiles['cfda_titles'].apply(parse_cfda_titles)

sample = org_profiles[org_profiles['num_programs'] >= 3].iloc[0]
sample[['org_name', 'agencies', 'cfda_programs', 'cfda_titles', 'num_programs']]

,13
org_name,UNIVERSITY OF NORTH CAROLINA AT GREENSBORO
agencies,"[Department of Education, Department of Health..."
cfda_programs,"[84.335, 93.31, 93.732, 81.049, 93.866, 47.076..."
cfda_titles,"[CHILD CARE ACCESS MEANS PARENTS IN SCHOOL, TR..."
num_programs,42


In [13]:
def simplify_org_type(raw):
    raw = str(raw).upper()
    if 'SMALL BUSINESS' in raw:
        return 'Small Business'
    if 'HIGHER EDUCATION' in raw:
        return 'University / Higher Education'
    if 'NONPROFIT' in raw:
        return 'Nonprofit'
    return 'Other'

org_profiles['org_type_simple'] = org_profiles['org_type'].apply(simplify_org_type)
org_profiles['org_type_simple'].value_counts()

,count
org_type_simple,
University / Higher Education,18346
Small Business,4765


In [14]:
grant_profiles[['cfda_number', 'cfda_title', 'num_recipients', 'avg_award', 'embedding_text_length']].head()

,cfda_number,cfda_title,num_recipients,avg_award,embedding_text_length
0,10.001,AGRICULTURAL RESEARCH BASIC AND APPLIED RESEARCH,131,216448.526311,29898
1,10.025,"PLANT AND ANIMAL DISEASE, PEST CONTROL, AND AN...",147,465767.200798,50095
2,10.028,WILDLIFE SERVICES,44,500054.932243,50057
3,10.047,TRIBAL FOOD SOVEREIGNTY,2,215925.000000,936
4,10.048,TRIBAL AGRICULTURE TECHNICAL ASSISTANCE,4,160275.000000,836


In [15]:
grant_profiles['embedding_text'].isna().sum()

np.int64(0)

In [16]:
org_profiles[['org_name', 'org_type_simple', 'state', 'num_programs', 'total_funding']].head()

,org_name,org_type_simple,state,num_programs,total_funding
0,EATON REGIONAL EDUCATION SERVICE AGENCY,University / Higher Education,MI,1,460000.0
1,"CAROLINA LAND AND LAKES RC-D, INC.",University / Higher Education,NC,1,1392285.0
2,PROVIDENCE HEALTH & SERVICES - OREGON,University / Higher Education,OR,2,1498815.0
3,UNIVERSITY OF ILLINOIS,University / Higher Education,IL,2,44088.1
4,COHERENT CORP,Small Business,PA,1,1050000.0


In [17]:
org_profiles['all_descriptions'].isna().sum()

np.int64(1)

In [18]:
org_profiles['all_descriptions'] = org_profiles['all_descriptions'].fillna('')

mask = org_profiles['org_type_simple'] == 'University / Higher Education'
wrong = org_profiles[mask & ~org_profiles['org_type'].str.contains('HIGHER EDUCATION', na=False)]
wrong[['org_name', 'org_type']].head(10)

,org_name,org_type


# Save to Drive

In [19]:
# Save outputs to Google Drive
grant_profiles.to_csv(f'{DRIVE_PATH}/grant_profiles.csv', index=False)
org_profiles.to_csv(f'{DRIVE_PATH}/org_profiles_clean.csv', index=False)